# Relevancy, Actor Extraction, & Stance Detection Pipeline (Full Dataset & Web-Ready)

Revisi produksi dari `Relevancy_Pipeline_NLP_News_Comment_FINALIZED.ipynb` untuk dijalankan di **Google Colab Cloud** pada **dataset penuh (`news_dataset.csv`, 20.205 artikel)** serta siap diintegrasikan ke website (`CapstoneRAP2026/integration/all`).

**Fitur Utama Revisi:**
1. **Full Dataset (`news_dataset.csv`)**: Mendukung dataset penuh tanpa kolom `no` / `Stance1` (evaluasi gold berjalan otomatis hanya jika ground-truth tersedia).
2. **Stage 1 Two-Stage Cascade**: Zero-token local regex pre-filter (`passes_local_domain_filter`) + kompresi konteks KWIC (`extract_condensed_context`, ~300 kata), menghemat ~61.6% panggilan API pada dataset penuh.
3. **Unlimited Rate-Limit Retry + Incrementing Pause**: Tidak pernah berhenti di `max_retries=3` atau menulis `Fallback error` palsu. Saat terkena limit (`429` / `RESOURCE_EXHAUSTED` / `quota`) atau error transien, jeda bertambah bertahap (`initial -> initial+step -> ... -> max_pause`) dan mencoba ulang tanpa batas hingga berhasil.
4. **Live Progress Bar (`tqdm.auto`) & Per-Batch Drive Checkpointing**: Menampilkan progres anotasi model secara real-time, menyimpan setiap batch langsung ke Google Drive (dengan fallback lokal), dan **mengisolasi checkpoint per `labeler_id`** (`model_name` + hash custom prompt) sehingga ganti model otomatis menjalankan prediksi ulang dari awal untuk model tersebut.
5. **Web-Ready Config (BYO Model, API Key, & Custom Prompt)**: Mendukung pemilihan model, API key pengguna, dan custom prompt yang dilapiskan di atas prompt dasar, serta mengekspor SQLite DB + Postgres staging CSV dengan stance huruf kecil (`support`, `oppose`, `neutral`).


## 1. Deteksi Lingkungan Colab, Dependensi, & Google Drive Mount


In [ ]:
import importlib.util
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
print(f"Lingkungan eksekusi: {'Google Colab Cloud' if IN_COLAB else 'Lokal / Jupyter'}")

# Cek dan install hanya library yang belum terpasang
REQUIRED_PACKAGES = {
    'google.genai': 'google-genai',
    'pydantic': 'pydantic',
    'tqdm': 'tqdm',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
}
missing = [pkg for mod, pkg in REQUIRED_PACKAGES.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"Menginstall paket yang belum tersedia: {missing}")
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print("Seluruh dependensi Python sudah tersedia.")

WORK_DIR = os.getcwd()
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        WORK_DIR = '/content/drive/MyDrive/CapstoneRAP_NewsPipeline'
        os.makedirs(WORK_DIR, exist_ok=True)
        print(f"Google Drive terpasang. Checkpoint & output disimpan ke: {WORK_DIR}")
    except Exception as e:
        print(f"[Peringatan] Gagal mount Google Drive ({e}). Menggunakan penyimpanan sesi lokal Colab.")
else:
    print(f"Direktori kerja lokal: {WORK_DIR}")


## 2. Konfigurasi Pipeline (Model, API Key, Custom Prompt, & Pause System)
Pengguna dapat memilih model sendiri, memasukkan API key sendiri, atau menambahkan custom prompt di atas prompt dasar kami. Mengubah model atau custom prompt akan menghasilkan `labeler_id` baru dan checkpoint terpisah.


In [ ]:
#@title Konfigurasi Model, Prompt Tambahan, & Rate-Limit Pause { display-mode: "form" }
import getpass
import hashlib
import re
from dataclasses import dataclass
from typing import Optional

MODEL_NAME = "gemini-3.1-flash-lite" #@param ["gemini-3.1-flash-lite", "gemma-4-26b-a4b-it", "gemini-2.5-flash", "gemini-2.5-flash-lite"] {allow-input: true}
INPUT_CSV_PATH = "news_dataset.csv" #@param {type:"string"}
GOLD_BENCHMARK_CSV = "news baru - News fixx.csv" #@param {type:"string"}

# Kosongkan jika hanya ingin menggunakan prompt dasar kami
CUSTOM_RELEVANCY_PROMPT = "" #@param {type:"string"}
CUSTOM_ACTOR_PROMPT = "" #@param {type:"string"}

USE_LOCAL_PREFILTER = True #@param {type:"boolean"}
USE_CONDENSED_CONTEXT = True #@param {type:"boolean"}
MAX_CONDENSED_WORDS = 300 #@param {type:"integer"}
RELEVANCY_BATCH_SIZE = 5 #@param {type:"integer"}
ACTOR_BATCH_SIZE = 2 #@param {type:"integer"}
TEMPERATURE = 0.0 #@param {type:"number"}

# Incrementing pause system (detik) saat terkena limit API (retry tanpa batas)
INITIAL_PAUSE_SEC = 15.0 #@param {type:"number"}
PAUSE_STEP_SEC = 15.0 #@param {type:"number"}
MAX_PAUSE_SEC = 300.0 #@param {type:"number"}
INTER_BATCH_DELAY_SEC = 1.0 #@param {type:"number"}
REQUEST_TIMEOUT_MS = 120000 #@param {type:"integer"}

# Opsional: isi langsung di runtime atau biarkan kosong untuk mengambil dari Colab Secrets / env / input aman
USER_API_KEY = ""

def resolve_api_key(explicit_key: Optional[str] = None) -> str:
    if explicit_key and explicit_key.strip():
        return explicit_key.strip()
    if IN_COLAB:
        try:
            from google.colab import userdata
            for name in ("GEMINI_API_KEY", "GEMMA", "GOOGLE_API_KEY"):
                try:
                    val = userdata.get(name)
                    if val and val.strip():
                        print(f"[Auth] Menggunakan API key dari Colab Secrets: {name}")
                        return val.strip()
                except Exception:
                    continue
        except Exception:
            pass
    for env_name in ("GEMINI_API_KEY", "GEMMA", "GOOGLE_API_KEY"):
        val = os.environ.get(env_name)
        if val and val.strip():
            print(f"[Auth] Menggunakan API key dari environment variable: {env_name}")
            return val.strip()
    entered = getpass.getpass("Masukkan Google AI Studio / Gemini API Key Anda: ").strip()
    if not entered:
        raise ValueError("API key wajib diisi untuk menjalankan prediksi model.")
    return entered

@dataclass
class PipelineConfig:
    model_name: str = MODEL_NAME
    api_key: Optional[str] = None
    custom_relevancy_prompt: str = CUSTOM_RELEVANCY_PROMPT
    custom_actor_prompt: str = CUSTOM_ACTOR_PROMPT
    use_local_prefilter: bool = USE_LOCAL_PREFILTER
    use_condensed_context: bool = USE_CONDENSED_CONTEXT
    max_condensed_words: int = MAX_CONDENSED_WORDS
    relevancy_batch_size: int = RELEVANCY_BATCH_SIZE
    actor_batch_size: int = ACTOR_BATCH_SIZE
    temperature: float = TEMPERATURE
    initial_pause_sec: float = INITIAL_PAUSE_SEC
    pause_step_sec: float = PAUSE_STEP_SEC
    max_pause_sec: float = MAX_PAUSE_SEC
    inter_batch_delay_sec: float = INTER_BATCH_DELAY_SEC
    request_timeout_ms: int = REQUEST_TIMEOUT_MS
    checkpoint_dir: str = os.path.join(WORK_DIR, "checkpoints")
    output_dir: str = os.path.join(WORK_DIR, "outputs")
    labeler_override: Optional[str] = None

    @property
    def has_custom_prompt(self) -> bool:
        return bool((self.custom_relevancy_prompt or "").strip() or (self.custom_actor_prompt or "").strip())

    @property
    def prompt_hash(self) -> str:
        if not self.has_custom_prompt:
            return "default"
        payload = f"{(self.custom_relevancy_prompt or '').strip()}\n---\n{(self.custom_actor_prompt or '').strip()}"
        return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:8]

    @property
    def labeler_id(self) -> str:
        if self.labeler_override:
            return re.sub(r"[^A-Za-z0-9._-]", "_", self.labeler_override.strip())
        base = re.sub(r"[^A-Za-z0-9._-]", "_", self.model_name.strip())
        return f"{base}__custom_{self.prompt_hash}" if self.has_custom_prompt else base

ACTIVE_API_KEY = resolve_api_key(USER_API_KEY)
CONFIG = PipelineConfig(api_key=ACTIVE_API_KEY)
print(f"Model Aktif : {CONFIG.model_name}")
print(f"Labeler ID  : {CONFIG.labeler_id} (checkpoint & output terisolasi per model/prompt)")
print(f"Checkpoint  : {os.path.join(CONFIG.checkpoint_dir, CONFIG.labeler_id)}")


## 3. Muat & Standarisasi Dataset Penuh (`news_dataset.csv`)
Mendukung pencarian file di direktori kerja, Google Drive, atau upload langsung di Colab, serta menormalisasi skema kolom tanpa bergantung pada kolom `no` atau `Stance1`.


In [ ]:
import pandas as pd
import numpy as np

def locate_input_file(path_hint: str) -> str:
    candidates = [
        path_hint,
        os.path.join(WORK_DIR, os.path.basename(path_hint)),
        os.path.join("/content", os.path.basename(path_hint)),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    if IN_COLAB:
        print(f"File '{path_hint}' belum ditemukan. Silakan upload file dataset CSV:")
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            return list(uploaded.keys())[0]
    raise FileNotFoundError(f"Dataset '{path_hint}' tidak ditemukan di {candidates}")

def find_column(df, candidates):
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

def load_and_standardize_dataset(path_hint: str):
    resolved_path = locate_input_file(path_hint)
    df_raw = pd.DataFrame()
    for enc in ['utf-8', 'latin1', 'cp1252']:
        try:
            df_try = pd.read_csv(resolved_path, encoding=enc)
            if df_try.shape[1] > 1 and len(df_try) > 0:
                df_raw = df_try
                break
        except Exception:
            continue
    if df_raw.empty:
        raise ValueError(f"Gagal membaca dataset dari {resolved_path}")

    df = df_raw.copy()
    df.columns = df.columns.astype(str).str.strip()

    title_col = find_column(df, ['title', 'judul', 'news_title'])
    content_col = find_column(df, ['content', 'isi', 'berita', 'article', 'text'])
    if not title_col or not content_col:
        raise ValueError(f"Kolom title/content tidak ditemukan: {list(df.columns)}")
    if title_col != 'title':
        df['title'] = df[title_col]
    if content_col != 'content':
        df['content'] = df[content_col]

    initial_len = len(df)
    df = df.dropna(subset=['title', 'content']).copy()
    df = df[df['title'].astype(str).str.strip().ne('') & df['content'].astype(str).str.strip().ne('')].reset_index(drop=True)

    df['row_id'] = df.index.astype(int)
    if 'no' not in df.columns:
        df['no'] = df['row_id'] + 1
    for col, default in [('source', 'unknown'), ('url', ''), ('keyword', ''), ('target', 'UMUM'), ('author', '')]:
        if col not in df.columns:
            df[col] = default
        else:
            df[col] = df[col].fillna(default).astype(str)

    parsed_dt = pd.to_datetime(df['date'], format='mixed', errors='coerce') if 'date' in df.columns else pd.Series([pd.NaT] * len(df))
    df['item_date'] = parsed_dt.dt.strftime('%Y-%m-%d').fillna('')
    df['publish_date'] = parsed_dt.dt.strftime('%Y-%m-%d %H:%M:%S').fillna('')
    df['publish_year'] = parsed_dt.dt.year.astype('Int64')
    df['publish_month'] = parsed_dt.dt.month.astype('Int64')

    has_gold = 'Stance1' in df.columns
    df['gold_relevancy'] = ((df['Stance1'] != 'Irrelevant') & (df['Stance1'].notna())) if has_gold else pd.NA
    return df, has_gold, resolved_path

df_matched, HAS_GOLD, RESOLVED_DATASET_PATH = load_and_standardize_dataset(CONFIG.INPUT_CSV_PATH if hasattr(CONFIG, 'INPUT_CSV_PATH') else INPUT_CSV_PATH)
print(f"[Sukses] Dataset: {RESOLVED_DATASET_PATH}")
print(f"Total artikel valid: {len(df_matched):,} | Gold annotation tersedia: {HAS_GOLD}")
display(df_matched[['row_id', 'doc_no' if 'doc_no' in df_matched.columns else 'no', 'source', 'target', 'item_date', 'title']].head(3))


## 4. Base Prompts, Layered Custom Prompt, Stage 1 Pre-Filter & KWIC Compression


In [ ]:
RELEVANCY_SYSTEM_INSTRUCTION = """Anda adalah analis kebijakan energi dan infrastruktur kelistrikan Indonesia tingkat pakar.
Tugas Anda adalah mengklasifikasikan apakah setiap artikel berita RELEVAN (relevancy = true) atau TIDAK RELEVAN (relevancy = false) terhadap:
- PLTU (Pembangkit Listrik Tenaga Uap / batubara) di Indonesia, ATAU
- PLTN (Pembangkit Listrik Tenaga Nuklir) di Indonesia.

PANDUAN KLASIFIKASI:
1. RELEVAN (relevancy = true): Artikel membahas operasional, kapasitas, emisi, polusi udara, co-firing biomassa, pasokan batubara, pensiun dini, transisi energi/bauran kelistrikan, atau rencana pembangunan PLTN/reaktor SMR di Indonesia.
2. TIDAK RELEVAN (relevancy = false): Peristiwa murni luar negeri tanpa kaitan kebijakan Indonesia, berita kriminal hukum umum di mana PLTU hanya disebut pasif sebagai barang bukti, atau berita lingkungan umum tanpa fokus pembangkit listrik.
"""

ACTOR_SYSTEM_INSTRUCTION = """Anda adalah sistem Information Extraction (IE) dan Argument Mining tingkat pakar untuk analisis kebijakan publik dan transisi energi di Indonesia (fokus PLTU Batubara vs PLTN Nuklir).
Tugas Anda adalah mengekstrak tokoh/aktor kunci beserta sikap (stance) dan bukti kutipan verbatimnya terhadap Target Fokus isu yang ditentukan untuk setiap dokumen (PLTU, PLTN, atau UMUM).

### PANDUAN OPERASIONAL & ATURAN EKSTRAKSI:
1. **Definisi Aktor**:
   - Tokoh perorangan, pejabat pemerintah, pimpinan BUMN/swasta, pakar/akademisi, aktivis LSM, atau saksi warga yang membuat pernyataan eksplisit terkait energi.
   - **KOLOM OPINI / TAJUK RENCANA**: Jika artikel merupakan opini/analisis (kolom opini), maka **penulis artikel tersebut** (yang namanya dan profilnya tercantum di byline atau akhir teks) adalah **Aktor Utama** dengan `speech_act_type: AUTHOR_OPINION`.
   - **Bukan Aktor**: Nama tokoh yang hanya disebut secara pasif sebagai objek pembicaraan orang lain.
   - **Konsolidasi Per Tokoh**: Untuk setiap tokoh unik, hasilkan tepat 1 entri yang memuat kutipan dan sikap paling representatif terhadap Target Fokus dokumen.

2. **Atribusi Tuturan (Speech Act & Verbatim Evidence)**:
   - `verbatim_evidence` WAJIB merupakan cuplikan teks ASLI yang ada di dalam artikel (substring persis). DILARANG memparafrase.
   - Pisahkan antara orang yang berbicara (Aktor) dan topik yang dibicarakan (Target).

3. **Klasifikasi Sikap (Target-Dependent Stance Classification)**:
   - Evaluasi sikap tokoh SECARA KHUSUS terhadap **Target Fokus Dokumen** yang diberikan:
     * `SUPPORT`: Mendukung eksistensi, investasi, kelanjutan operasi, atau teknologi PLTU/PLTN terkait.
     * `OPPOSE`: Mengkritik, memprotes dampak polusi/lingkungan, menolak proyek, atau mendesak penghentian/pensiun dini.
     * `NEUTRAL`: Bersifat prosedural, netral, regulatori, informatif, atau data faktual tanpa keberpihakan normatif yang jelas.

4. Jika artikel tidak memuat pernyataan sikap dari tokoh perorangan, kembalikan `actors: []`.
"""

ENERGY_PREFILTER_KEYWORDS = [
    r'\bpltu\b', r'\bpltn\b',
    r'pembangkit\s+(?:listrik|daya|batubara|nuklir)?',
    r'tenaga\s+uap', r'tenaga\s+nuklir',
    r'reaktor\s+(?:nuklir|daya|modular|smr|kartini|serpong)',
    r'\bsmr\b', r'thorium',
    r'batu\s*bara',
    r'co-firing', r'cofiring',
    r'pensiun\s+dini',
    r'transisi\s+energi', r'bauran\s+energi',
    r'\bjetp\b', r'\betm\b',
    r'\bbatan\b', r'\bbapeten\b',
]
ENERGY_PREFILTER_REGEX = re.compile('|'.join(ENERGY_PREFILTER_KEYWORDS), re.IGNORECASE)

def passes_local_domain_filter(title: str, content: str) -> bool:
    return bool(ENERGY_PREFILTER_REGEX.search(f"{title or ''} {content or ''}"))

def split_into_sentences(text: str):
    clean = re.sub(r'\r\n|\r', '\n', str(text or ''))
    sentences = []
    for p in [p.strip() for p in clean.split('\n') if p.strip()]:
        for s in re.split(r'(?<=[.!?])\s+', p):
            s_clean = s.strip()
            if len(s_clean) > 10:
                sentences.append(s_clean)
    return sentences

def extract_condensed_context(title: str, content: str, max_words: int = 300) -> str:
    title_str = str(title or '').strip()
    sentences = split_into_sentences(content)
    if not sentences:
        fallback = str(content or '')[: max_words * 6]
        return f"{title_str}\n{fallback}" if title_str else fallback

    selected = set(range(min(3, len(sentences))))
    for i, s in enumerate(sentences):
        if ENERGY_PREFILTER_REGEX.search(s):
            selected.update([max(0, i - 1), i, min(len(sentences) - 1, i + 1)])

    extracted = []
    word_count = len(title_str.split())
    for idx in sorted(selected):
        s = sentences[idx]
        w_len = len(s.split())
        if word_count + w_len > max_words:
            rem = max_words - word_count
            if rem > 10:
                extracted.append(' '.join(s.split()[:rem]) + '...')
            break
        extracted.append(s)
        word_count += w_len

    body = '\n'.join(extracted)
    return f"{title_str}\n{body}" if title_str else body

def compose_system_instruction(base_instruction: str, custom_prompt: str = "") -> str:
    extra = (custom_prompt or "").strip()
    if not extra:
        return base_instruction
    return (
        f"{base_instruction.rstrip()}\n\n"
        "=== INSTRUKSI TAMBAHAN PENGGUNA (ON TOP OF BASE RUBRIC) ===\n"
        f"{extra}\n"
        "Catatan: Tetap patuhi struktur JSON schema dan ID dokumen input yang diminta.\n"
    )

REL_SYS_PROMPT = compose_system_instruction(RELEVANCY_SYSTEM_INSTRUCTION, CONFIG.custom_relevancy_prompt)
ACT_SYS_PROMPT = compose_system_instruction(ACTOR_SYSTEM_INSTRUCTION, CONFIG.custom_actor_prompt)

if CONFIG.use_local_prefilter:
    n_pass = sum(passes_local_domain_filter(t, c) for t, c in zip(df_matched['title'], df_matched['content']))
    print(f"[Stage 1 Pre-Filter] Lolos ke LLM: {n_pass:,} / {len(df_matched):,} ({n_pass/len(df_matched)*100:.1f}%) | Disaring lokal (0 token): {len(df_matched)-n_pass:,}")


## 5. Pydantic Schemas, Tolerant JSON Parser, & Unlimited Incrementing-Pause Retry Engine


In [ ]:
import json
import time
from typing import List, Literal, Dict, Any, Sequence
from pydantic import BaseModel, Field
from google import genai
from google.genai import types
from tqdm.auto import tqdm

class RelevancyItem(BaseModel):
    id: int = Field(description='ID baris dokumen artikel berita.')
    relevancy: bool = Field(description='True jika relevan terhadap PLTU/PLTN di Indonesia; False jika tidak.')
    confidence: float = Field(description='Tingkat keyakinan model antara 0.0 hingga 1.0.', ge=0.0, le=1.0)
    reasoning: str = Field(description='Penjelasan ringkas 1 kalimat mengapa artikel relevan atau tidak relevan.')

class RelevancyBatchResponse(BaseModel):
    results: List[RelevancyItem]

class DetectedActor(BaseModel):
    actor_name: str = Field(description='Nama lengkap tokoh atau individu yang dikutip atau menyatakan pandangan.')
    role_or_affiliation: str = Field(description='Jabatan, institusi, atau peran tokoh.')
    target_topic: Literal['PLTU', 'PLTN', 'UMUM'] = Field(description="Target fokus: 'PLTU', 'PLTN', atau 'UMUM'.")
    speech_act_type: Literal['DIRECT_QUOTE', 'INDIRECT_SPEECH', 'AUTHOR_OPINION'] = Field(description='Bentuk tuturan.')
    verbatim_evidence: str = Field(description='Kutipan langsung persis (verbatim quote) dari teks artikel asli.')
    stance: Literal['SUPPORT', 'OPPOSE', 'NEUTRAL'] = Field(description="Sikap tokoh: 'SUPPORT', 'OPPOSE', atau 'NEUTRAL'.")
    rationale: str = Field(description='Penjelasan ringkas 1 kalimat mengapa bukti mencerminkan sikap.')

class ArticleActorDetection(BaseModel):
    id: int = Field(description='ID baris dokumen artikel.')
    actors: List[DetectedActor] = Field(default_factory=list)

class BatchActorDetectionResponse(BaseModel):
    results: List[ArticleActorDetection]

def resolve_evidence_span(content: str, evidence: str) -> Dict[str, Any]:
    if not evidence or not content:
        return {'start_char': -1, 'end_char': -1, 'is_grounded': False}
    clean_ev = evidence.strip(' "“\'”«»').strip()
    if not clean_ev:
        return {'start_char': -1, 'end_char': -1, 'is_grounded': False}
    idx = content.find(clean_ev)
    if idx != -1:
        return {'start_char': idx, 'end_char': idx + len(clean_ev), 'is_grounded': True}
    lower_idx = content.lower().find(clean_ev.lower())
    if lower_idx != -1:
        return {'start_char': lower_idx, 'end_char': lower_idx + len(clean_ev), 'is_grounded': True}
    normalized_pattern = re.escape(re.sub(r'\s+', ' ', clean_ev))
    pattern = re.sub(r'\\ ', r'\\s+', normalized_pattern)
    match = re.search(pattern, content, re.IGNORECASE)
    if match:
        return {'start_char': match.start(), 'end_char': match.end(), 'is_grounded': True}
    leading_anchor = clean_ev[:35].strip()
    if len(leading_anchor) >= 15:
        l_idx = content.lower().find(leading_anchor.lower())
        if l_idx != -1:
            return {'start_char': l_idx, 'end_char': min(len(content), l_idx + len(clean_ev)), 'is_grounded': True}
    return {'start_char': -1, 'end_char': -1, 'is_grounded': False}

FENCE_RE = re.compile(r'```(?:json)?', re.IGNORECASE)
_FATAL_RE = re.compile(r'API_KEY_INVALID|API key not valid|UNAUTHENTICATED|PERMISSION_DENIED|\b401\b|\b403\b|\b404\b|NOT_FOUND', re.IGNORECASE)
_RATE_LIMIT_RE = re.compile(r'429|RESOURCE_EXHAUSTED|quota|rate.?limit|too many requests', re.IGNORECASE)
_RETRY_DELAY_RE = re.compile(r'retry(?:\s+in|\s+after|Delay["\':\s]+)\s*(\d+(?:\.\d+)?)\s*s', re.IGNORECASE)
_SCHEMA_UNSUPPORTED_RE = re.compile(r'response_schema|response_mime_type|json mode|mime', re.IGNORECASE)

class FatalPipelineError(RuntimeError):
    pass

def _extract_json_candidates(text: str):
    if not text:
        return []
    cleaned = FENCE_RE.sub('', text).strip()
    dec = json.JSONDecoder()
    out = []
    for m in re.finditer(r'[\[{]', cleaned):
        try:
            obj, _ = dec.raw_decode(cleaned[m.start():])
            out.append(obj)
        except json.JSONDecodeError:
            continue
    return out

def _coerce_bool(val):
    if isinstance(val, bool):
        return val
    if isinstance(val, (int, float)):
        return bool(val)
    if isinstance(val, str):
        s = val.strip().lower()
        if s in {'true', '1', 'yes', 'ya', 'relevan', 'relevant'}:
            return True
        if s in {'false', '0', 'no', 'tidak', 'tidak relevan', 'irrelevant'}:
            return False
    return None

def parse_relevancy_response(text: str, expected_ids: Sequence[int]) -> Dict[int, RelevancyItem]:
    expected_set = set(expected_ids)
    try:
        parsed = RelevancyBatchResponse.model_validate_json(text)
        out = {x.id: x for x in parsed.results if x.id in expected_set}
        if out:
            return out
    except Exception:
        pass
    for cand in _extract_json_candidates(text):
        items = cand.get('results', cand.get('hasil', [])) if isinstance(cand, dict) else cand
        if isinstance(cand, dict) and 'id' in cand and not items:
            items = [cand]
        if not isinstance(items, list):
            continue
        out = {}
        for it in items:
            if not isinstance(it, dict):
                continue
            try:
                rid = int(it.get('id'))
            except (TypeError, ValueError):
                continue
            if rid not in expected_set:
                continue
            rel = _coerce_bool(it.get('relevancy', it.get('relevant', it.get('relevan'))))
            if rel is None:
                continue
            try:
                conf = max(0.0, min(1.0, float(it.get('confidence', 0.8))))
            except (TypeError, ValueError):
                conf = 0.8
            out[rid] = RelevancyItem(id=rid, relevancy=rel, confidence=conf, reasoning=str(it.get('reasoning', it.get('alasan', ''))).strip())
        if out:
            return out
    return {}

def parse_actors_response(text: str, expected_ids: Sequence[int]) -> Dict[int, ArticleActorDetection]:
    expected_set = set(expected_ids)
    try:
        parsed = BatchActorDetectionResponse.model_validate_json(text)
        out = {x.id: x for x in parsed.results if x.id in expected_set}
        if out:
            return out
    except Exception:
        pass
    t_map = {'PLTU': 'PLTU', 'PLTN': 'PLTN', 'UMUM': 'UMUM', 'GENERAL': 'UMUM'}
    s_map = {'DIRECT_QUOTE': 'DIRECT_QUOTE', 'INDIRECT_SPEECH': 'INDIRECT_SPEECH', 'AUTHOR_OPINION': 'AUTHOR_OPINION'}
    st_map = {'SUPPORT': 'SUPPORT', 'PRO': 'SUPPORT', 'MENDUKUNG': 'SUPPORT', 'OPPOSE': 'OPPOSE', 'AGAINST': 'OPPOSE', 'MENOLAK': 'OPPOSE', 'NEUTRAL': 'NEUTRAL', 'NETRAL': 'NEUTRAL'}
    for cand in _extract_json_candidates(text):
        items = cand.get('results', cand.get('hasil', [])) if isinstance(cand, dict) else cand
        if isinstance(cand, dict) and 'id' in cand and not items:
            items = [cand]
        if not isinstance(items, list):
            continue
        out = {}
        for it in items:
            if not isinstance(it, dict):
                continue
            try:
                rid = int(it.get('id'))
            except (TypeError, ValueError):
                continue
            if rid not in expected_set or not isinstance(it.get('actors', []), list):
                continue
            actors = []
            for a in it.get('actors', []):
                if not isinstance(a, dict) or not str(a.get('actor_name', '')).strip():
                    continue
                actors.append(DetectedActor(
                    actor_name=str(a['actor_name']).strip(),
                    role_or_affiliation=str(a.get('role_or_affiliation', '')).strip(),
                    target_topic=t_map.get(str(a.get('target_topic', '')).strip().upper(), 'UMUM'),
                    speech_act_type=s_map.get(str(a.get('speech_act_type', '')).strip().upper(), 'INDIRECT_SPEECH'),
                    verbatim_evidence=str(a.get('verbatim_evidence', '')).strip(),
                    stance=st_map.get(str(a.get('stance', '')).strip().upper(), 'NEUTRAL'),
                    rationale=str(a.get('rationale', '')).strip(),
                ))
            out[rid] = ArticleActorDetection(id=rid, actors=actors)
        if out:
            return out
    return {}

class GenAIRateLimitClient:
    def __init__(self, cfg: PipelineConfig):
        self.cfg = cfg
        self.client = genai.Client(api_key=cfg.api_key, http_options=types.HttpOptions(timeout=cfg.request_timeout_ms))
        self.supports_structured = True

    def generate(self, system_instruction: str, user_prompt: str, response_schema: Any) -> str:
        prompt = user_prompt
        cfg_kwargs = {'system_instruction': system_instruction, 'temperature': self.cfg.temperature}
        if self.supports_structured and response_schema is not None:
            cfg_kwargs['response_mime_type'] = 'application/json'
            cfg_kwargs['response_schema'] = response_schema
        else:
            schema_json = json.dumps(response_schema.model_json_schema(), ensure_ascii=False)
            prompt = f"{user_prompt}\n\nKEMBALIKAN HANYA JSON VALID tanpa teks lain sesuai skema:\n{schema_json}"
        try:
            resp = self.client.models.generate_content(
                model=self.cfg.model_name, contents=prompt, config=types.GenerateContentConfig(**cfg_kwargs)
            )
            return resp.text or ''
        except Exception as e:
            msg = str(e)
            if _FATAL_RE.search(msg):
                raise FatalPipelineError(msg[:400]) from e
            if self.supports_structured and _SCHEMA_UNSUPPORTED_RE.search(msg) and not _RATE_LIMIT_RE.search(msg):
                self.supports_structured = False
                return self.generate(system_instruction, user_prompt, response_schema)
            raise

def call_with_unlimited_retry(client: GenAIRateLimitClient, system_instruction: str, user_prompt: str, schema: Any, parser_fn, expected_ids: Sequence[int], pbar=None):
    consecutive_hits = 0
    while True:
        try:
            raw_text = client.generate(system_instruction, user_prompt, schema)
            parsed = parser_fn(raw_text, expected_ids)
            if parsed:
                return parsed
            raise ValueError(f"JSON tidak memuat ID {list(expected_ids)}: {raw_text[:120]!r}")
        except FatalPipelineError:
            raise
        except Exception as e:
            if _FATAL_RE.search(str(e)):
                raise FatalPipelineError(str(e)[:400]) from e
            base_pause = min(CONFIG.initial_pause_sec + consecutive_hits * CONFIG.pause_step_sec, CONFIG.max_pause_sec)
            m = _RETRY_DELAY_RE.search(str(e))
            wait_sec = max(base_pause, min(float(m.group(1)) + 2.0, CONFIG.max_pause_sec)) if m else base_pause
            consecutive_hits += 1
            reason = "Rate limit/quota" if _RATE_LIMIT_RE.search(str(e)) else type(e).__name__
            remaining = float(wait_sec)
            while remaining > 0:
                if pbar is not None:
                    pbar.set_postfix_str(f"{reason} | retry #{consecutive_hits} | wait {int(round(remaining))}s")
                step = min(5.0, remaining)
                time.sleep(step)
                remaining -= step
            if pbar is not None:
                pbar.set_postfix_str(f"retrying #{consecutive_hits}...")

CLIENT = GenAIRateLimitClient(CONFIG)
print(f"GenAIRateLimitClient siap untuk model '{CONFIG.model_name}' (retry tanpa batas aktif).")


## 6. Tahap 1: Klasifikasi Relevansi dengan Progress Bar & Checkpoint Per-Batch


In [ ]:
CHECKPOINT_ROOT = os.path.join(CONFIG.checkpoint_dir, CONFIG.labeler_id)
os.makedirs(CHECKPOINT_ROOT, exist_ok=True)
RELEVANCY_CHECKPOINT_FILE = os.path.join(CHECKPOINT_ROOT, "relevancy_checkpoint.csv")

def process_relevancy_batch(batch_df: pd.DataFrame, pbar) -> Dict[int, RelevancyItem]:
    parts = ["Analisis relevansi artikel-artikel berita berikut terhadap isu PLTU/PLTN di Indonesia:\n"]
    for _, row in batch_df.iterrows():
        rid = int(row['row_id'])
        title = str(row['title'])
        raw_content = str(row['content'])
        body = extract_condensed_context(title, raw_content, max_words=CONFIG.max_condensed_words) if CONFIG.use_condensed_context else raw_content[:4500]
        parts.append(f"--- DOKUMEN ID: {rid} ---\nJudul: {title}\nIsi Ringkas:\n{body}\n")
    expected_ids = [int(x) for x in batch_df['row_id'].tolist()]
    parsed = call_with_unlimited_retry(CLIENT, REL_SYS_PROMPT, "\n".join(parts), RelevancyBatchResponse, parse_relevancy_response, expected_ids, pbar=pbar)
    missing = [rid for rid in expected_ids if rid not in parsed]
    if missing and len(batch_df) > 1:
        for rid in missing:
            parsed.update(process_relevancy_batch(batch_df[batch_df['row_id'] == rid], pbar))
    return parsed

relevancy_map = {}
if os.path.exists(RELEVANCY_CHECKPOINT_FILE):
    df_rcp = pd.read_csv(RELEVANCY_CHECKPOINT_FILE)
    for _, r in df_rcp.iterrows():
        relevancy_map[int(r['row_id'])] = {
            'relevancy': bool(r['relevancy']),
            'confidence': float(r['confidence']),
            'reasoning': str(r.get('reasoning', '')),
            'filtered_by': str(r.get('filtered_by', 'STAGE_2_GEMINI')),
        }
    print(f"[Resume Stage 1] Memuat {len(relevancy_map):,} artikel dari {RELEVANCY_CHECKPOINT_FILE}")

df_rel_todo = df_matched[~df_matched['row_id'].isin(relevancy_map.keys())].copy()
with tqdm(total=len(df_matched), initial=len(relevancy_map), desc=f"[Stage 1 Relevancy | {CONFIG.labeler_id}]", unit="art") as pbar:
    if len(df_rel_todo) > 0:
        local_rows = []
        llm_indices = []
        for idx, row in df_rel_todo.iterrows():
            rid = int(row['row_id'])
            if CONFIG.use_local_prefilter and not passes_local_domain_filter(str(row['title']), str(row['content'])):
                rec = {
                    'relevancy': False,
                    'confidence': 1.0,
                    'reasoning': 'Local Domain Pre-filter: Tidak memuat entitas atau terminologi PLTU/PLTN/Energi.',
                    'filtered_by': 'STAGE_1_LOCAL',
                }
                relevancy_map[rid] = rec
                local_rows.append({'row_id': rid, **rec})
            else:
                llm_indices.append(idx)

        if local_rows:
            pd.DataFrame(local_rows).to_csv(RELEVANCY_CHECKPOINT_FILE, mode='a', index=False, header=not os.path.exists(RELEVANCY_CHECKPOINT_FILE), encoding='utf-8')
            pbar.update(len(local_rows))

        if llm_indices:
            cand_df = df_rel_todo.loc[llm_indices]
            bsize = max(1, CONFIG.relevancy_batch_size)
            for i in range(0, len(cand_df), bsize):
                b_df = cand_df.iloc[i : i + bsize]
                parsed = process_relevancy_batch(b_df, pbar)
                batch_rows = []
                for rid, item in parsed.items():
                    rec = {'relevancy': bool(item.relevancy), 'confidence': float(item.confidence), 'reasoning': str(item.reasoning), 'filtered_by': 'STAGE_2_GEMINI'}
                    relevancy_map[rid] = rec
                    batch_rows.append({'row_id': rid, **rec})
                if batch_rows:
                    pd.DataFrame(batch_rows).to_csv(RELEVANCY_CHECKPOINT_FILE, mode='a', index=False, header=not os.path.exists(RELEVANCY_CHECKPOINT_FILE), encoding='utf-8')
                pbar.set_postfix_str("OK")
                pbar.update(len(b_df))
                if CONFIG.inter_batch_delay_sec > 0 and (i + bsize) < len(cand_df):
                    time.sleep(CONFIG.inter_batch_delay_sec)

df_matched['pred_relevancy'] = df_matched['row_id'].map(lambda x: relevancy_map.get(x, {}).get('relevancy', False))
df_matched['relevancy_confidence'] = df_matched['row_id'].map(lambda x: relevancy_map.get(x, {}).get('confidence', 0.0))
df_matched['relevancy_reasoning'] = df_matched['row_id'].map(lambda x: relevancy_map.get(x, {}).get('reasoning', ''))
df_matched['relevancy_filter_tier'] = df_matched['row_id'].map(lambda x: relevancy_map.get(x, {}).get('filtered_by', 'UNKNOWN'))
print(f"\n[Stage 1 Selesai] Distribusi Prediksi Relevansi: {df_matched['pred_relevancy'].value_counts().to_dict()}")


## 7. Tahap 2: Ekstraksi Aktor, Kutipan Verbatim, & Stance (`pred_relevancy == True`)


In [ ]:
ACTORS_CHECKPOINT_FILE = os.path.join(CHECKPOINT_ROOT, "actors_checkpoint.csv")

def process_actors_batch(batch_df: pd.DataFrame, pbar) -> Dict[int, ArticleActorDetection]:
    parts = ["Berikut adalah teks artikel berita. Ekstrak tokoh kunci beserta bukti kutipan verbatim dan sikap (stance) terhadap Target Fokus yang ditentukan:\n"]
    for _, row in batch_df.iterrows():
        rid = int(row['row_id'])
        target_fokus = str(row.get('target', 'UMUM')).strip() or 'UMUM'
        title = str(row['title'])
        author = str(row.get('author', '')).strip()
        author_line = f"Penulis/Byline: {author}\n" if author else ""
        content = str(row['content'])[:8500]
        parts.append(f"--- DOKUMEN ID: {rid} ---\nTarget Fokus Isu: {target_fokus}\n{author_line}Judul: {title}\nIsi Teks:\n{content}\n")
    expected_ids = [int(x) for x in batch_df['row_id'].tolist()]
    parsed = call_with_unlimited_retry(CLIENT, ACT_SYS_PROMPT, "\n".join(parts), BatchActorDetectionResponse, parse_actors_response, expected_ids, pbar=pbar)
    missing = [rid for rid in expected_ids if rid not in parsed]
    if missing and len(batch_df) > 1:
        for rid in missing:
            parsed.update(process_actors_batch(batch_df[batch_df['row_id'] == rid], pbar))
    return parsed

actor_preds = {}
if os.path.exists(ACTORS_CHECKPOINT_FILE):
    df_acp = pd.read_csv(ACTORS_CHECKPOINT_FILE)
    for rid_val, grp in df_acp.groupby('row_id'):
        rid = int(rid_val)
        actor_preds[rid] = []
        for _, r in grp.iterrows():
            name = str(r.get('actor_name', '')).strip() if pd.notna(r.get('actor_name')) else ''
            if name:
                actor_preds[rid].append({
                    'actor_name': name,
                    'role_or_affiliation': str(r.get('role_or_affiliation', '')).strip() if pd.notna(r.get('role_or_affiliation')) else '',
                    'target_topic': str(r.get('target_topic', '')).strip() if pd.notna(r.get('target_topic')) else '',
                    'speech_act_type': str(r.get('speech_act_type', '')).strip() if pd.notna(r.get('speech_act_type')) else '',
                    'verbatim_evidence': str(r.get('verbatim_evidence', '')).strip() if pd.notna(r.get('verbatim_evidence')) else '',
                    'start_char': int(r['start_char']) if pd.notna(r.get('start_char')) else -1,
                    'end_char': int(r['end_char']) if pd.notna(r.get('end_char')) else -1,
                    'stance': str(r.get('stance', '')).strip().lower() if pd.notna(r.get('stance')) else '',
                    'rationale': str(r.get('rationale', '')).strip() if pd.notna(r.get('rationale')) else '',
                })
    print(f"[Resume Stage 2] Memuat {len(actor_preds):,} artikel relevan dari {ACTORS_CHECKPOINT_FILE}")

df_candidate_actors = df_matched[df_matched['pred_relevancy'] == True].copy()
df_actors_todo = df_candidate_actors[~df_candidate_actors['row_id'].isin(actor_preds.keys())].copy()
already_act = len(df_candidate_actors) - len(df_actors_todo)

with tqdm(total=len(df_candidate_actors), initial=already_act, desc=f"[Stage 2 Actors | {CONFIG.labeler_id}]", unit="art") as pbar:
    if len(df_actors_todo) > 0:
        bsize = max(1, CONFIG.actor_batch_size)
        for i in range(0, len(df_actors_todo), bsize):
            b_df = df_actors_todo.iloc[i : i + bsize]
            parsed = process_actors_batch(b_df, pbar)
            cp_rows = []
            for _, row in b_df.iterrows():
                rid = int(row['row_id'])
                content = str(row['content'])
                det = parsed.get(rid, ArticleActorDetection(id=rid, actors=[]))
                actor_preds[rid] = []
                if not det.actors:
                    cp_rows.append({
                        'row_id': rid, 'actor_name': '', 'role_or_affiliation': '', 'target_topic': '',
                        'speech_act_type': '', 'verbatim_evidence': '', 'start_char': -1, 'end_char': -1,
                        'stance': '', 'rationale': '',
                    })
                else:
                    for act in det.actors:
                        span = resolve_evidence_span(content, act.verbatim_evidence)
                        rec = {
                            'actor_name': act.actor_name,
                            'role_or_affiliation': act.role_or_affiliation,
                            'target_topic': act.target_topic,
                            'speech_act_type': act.speech_act_type,
                            'verbatim_evidence': act.verbatim_evidence,
                            'start_char': span['start_char'],
                            'end_char': span['end_char'],
                            'stance': act.stance.lower(),
                            'rationale': act.rationale,
                        }
                        actor_preds[rid].append(rec)
                        cp_rows.append({'row_id': rid, **rec})
            if cp_rows:
                pd.DataFrame(cp_rows).to_csv(ACTORS_CHECKPOINT_FILE, mode='a', index=False, header=not os.path.exists(ACTORS_CHECKPOINT_FILE), encoding='utf-8')
            pbar.set_postfix_str("OK")
            pbar.update(len(b_df))
            if CONFIG.inter_batch_delay_sec > 0 and (i + bsize) < len(df_actors_todo):
                time.sleep(CONFIG.inter_batch_delay_sec)

print(f"\n[Stage 2 Selesai] Total artikel relevan terproses: {len(df_candidate_actors):,}")


## 8. Evaluasi Opsional terhadap Benchmark Gold (Bila Tersedia)


In [ ]:
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

if HAS_GOLD:
    y_true = df_matched['gold_relevancy'].astype(bool).values
    y_pred = df_matched['pred_relevancy'].astype(bool).values
    cm = confusion_matrix(y_true, y_pred, labels=[False, True])
    display(pd.DataFrame(cm, index=['True IRRELEVANT', 'True RELEVANT'], columns=['Pred IRRELEVANT', 'Pred RELEVANT']))
    print(f"Precision: {precision_score(y_true, y_pred, zero_division=0)*100:.2f}% | Recall: {recall_score(y_true, y_pred, zero_division=0)*100:.2f}% | Macro-F1: {f1_score(y_true, y_pred, zero_division=0):.4f}")
elif os.path.exists(GOLD_BENCHMARK_CSV):
    df_gold = pd.read_csv(GOLD_BENCHMARK_CSV)
    df_gold['gold_rel'] = (df_gold['Stance1'] != 'Irrelevant') & (df_gold['Stance1'].notna())
    merged_eval = df_gold.merge(df_matched[['url', 'pred_relevancy']].drop_duplicates('url'), on='url', how='inner')
    if len(merged_eval) > 0:
        cm = confusion_matrix(merged_eval['gold_rel'], merged_eval['pred_relevancy'], labels=[False, True])
        print(f"Cross-check terhadap {len(merged_eval)} artikel gold (berdasarkan URL):")
        display(pd.DataFrame(cm, index=['True IRRELEVANT', 'True RELEVANT'], columns=['Pred IRRELEVANT', 'Pred RELEVANT']))
    else:
        print("Dataset gold ditemukan tetapi tidak ada URL yang cocok dengan dataset aktif.")
else:
    print("[Info] Evaluasi gold dilewati karena dataset penuh tidak memiliki kolom anotasi manusia.")


## 9. Konsolidasi & Ekspor Output Web-Ready (Master CSV, SQLite DB, & Postgres Staging CSV)


In [ ]:
import sqlite3
from datetime import datetime, timezone

OUTPUT_ROOT = os.path.join(CONFIG.output_dir, CONFIG.labeler_id)
os.makedirs(OUTPUT_ROOT, exist_ok=True)

master_rows = []
stmt_id = 1
for _, row in df_matched.iterrows():
    rid = int(row['row_id'])
    is_rel = bool(row['pred_relevancy'])
    preds = actor_preds.get(rid, []) if is_rel else []
    base = {
        'statement_id': stmt_id,
        'article_id': rid + 1,
        'row_id': rid,
        'doc_no': int(row['no']),
        'labeler': CONFIG.labeler_id,
        'model_name': CONFIG.model_name,
        'platform': 'News',
        'source': str(row['source']),
        'target_energy': str(row['target']).upper(),
        'target_code': str(row['target']).upper(),
        'item_date': str(row['item_date']),
        'publish_date': str(row['publish_date']),
        'publish_year': None if pd.isna(row['publish_year']) else int(row['publish_year']),
        'publish_month': None if pd.isna(row['publish_month']) else int(row['publish_month']),
        'url': str(row['url']),
        'title': str(row['title']),
        'keyword': str(row['keyword']),
        'is_relevant_pred': is_rel,
        'relevancy_confidence': float(row['relevancy_confidence']),
        'relevancy_reasoning': str(row['relevancy_reasoning']),
        'relevancy_filter_tier': str(row['relevancy_filter_tier']),
        'is_relevant_gold': None if pd.isna(row['gold_relevancy']) else bool(row['gold_relevancy']),
    }
    if not preds:
        master_rows.append({
            **base,
            'actor_name': '', 'role_or_affiliation': '', 'target_topic': '', 'speech_act_type': '',
            'verbatim_evidence': '', 'start_char': None, 'end_char': None, 'stance': '', 'stance_rationale': '',
        })
        stmt_id += 1
    else:
        for act in preds:
            master_rows.append({
                **base,
                'statement_id': stmt_id,
                'actor_name': act['actor_name'],
                'role_or_affiliation': act['role_or_affiliation'],
                'target_topic': act['target_topic'],
                'speech_act_type': act['speech_act_type'],
                'verbatim_evidence': act['verbatim_evidence'],
                'start_char': None if act['start_char'] < 0 else int(act['start_char']),
                'end_char': None if act['end_char'] < 0 else int(act['end_char']),
                'stance': act['stance'].lower(),
                'stance_rationale': act['rationale'],
            })
            stmt_id += 1

df_master = pd.DataFrame(master_rows)
master_csv_path = os.path.join(OUTPUT_ROOT, f"news_pipeline_master_output__{CONFIG.labeler_id}.csv")
df_master.to_csv(master_csv_path, index=False, encoding='utf-8')

article_rows = []
for aid, grp in df_master.groupby('article_id', sort=False):
    first = grp.iloc[0]
    article_rows.append({
        'article_id': int(aid),
        'doc_no': int(first['doc_no']),
        'labeler': CONFIG.labeler_id,
        'platform': 'News',
        'source': first['source'],
        'target_energy': first['target_energy'],
        'target_code': first['target_code'],
        'item_date': first['item_date'],
        'publish_date': first['publish_date'],
        'publish_year': first['publish_year'],
        'publish_month': first['publish_month'],
        'title': first['title'],
        'url': first['url'],
        'keyword': first['keyword'],
        'is_relevant': bool(first['is_relevant_pred']),
        'relevancy_confidence': float(first['relevancy_confidence']),
        'relevancy_reasoning': first['relevancy_reasoning'],
        'relevancy_filter_tier': first['relevancy_filter_tier'],
        'total_actors_detected': int(grp['actor_name'].fillna('').str.strip().ne('').sum()),
    })
df_articles = pd.DataFrame(article_rows)
df_stances = df_master[[
    'statement_id', 'article_id', 'doc_no', 'labeler', 'platform', 'target_energy',
    'item_date', 'is_relevant_pred', 'actor_name', 'role_or_affiliation', 'target_topic',
    'speech_act_type', 'verbatim_evidence', 'start_char', 'end_char', 'stance', 'stance_rationale'
]].rename(columns={'is_relevant_pred': 'is_relevant'})
df_labelers = pd.DataFrame([{
    'labeler': CONFIG.labeler_id,
    'labeler_type': 'llm',
    'task': 'news_relevancy_actor_stance',
    'model_name': CONFIG.model_name,
    'has_custom_prompt': CONFIG.has_custom_prompt,
    'prompt_hash': CONFIG.prompt_hash,
    'created_at': datetime.now(timezone.utc).isoformat(),
}])

db_path = os.path.join(OUTPUT_ROOT, f"news_stance_database__{CONFIG.labeler_id}.db")
if os.path.exists(db_path):
    os.remove(db_path)
conn = sqlite3.connect(db_path)
df_labelers.to_sql('labelers', conn, if_exists='replace', index=False)
df_articles.to_sql('articles', conn, if_exists='replace', index=False)
df_stances.to_sql('actor_stances', conn, if_exists='replace', index=False)
df_master.to_sql('master_news_stance', conn, if_exists='replace', index=False)
conn.execute('CREATE INDEX idx_master_filters ON master_news_stance(labeler, target_energy, platform, item_date);')
conn.execute('CREATE INDEX idx_articles_filters ON articles(labeler, target_energy, platform, item_date);')
conn.execute('CREATE INDEX idx_stances_filters ON actor_stances(labeler, target_energy, platform, item_date);')
conn.execute('''
    CREATE VIEW v_analysis_detail AS
    SELECT statement_id, article_id, doc_no, labeler, model_name, platform, source,
           target_energy, target_code, item_date, publish_date, publish_year, publish_month,
           title, url, keyword, is_relevant_pred AS is_relevant, actor_name,
           role_or_affiliation, target_topic, stance, verbatim_evidence
    FROM master_news_stance;
''')
conn.commit()
conn.close()

df_labelers.to_csv(os.path.join(OUTPUT_ROOT, 'ref_labeler.csv'), index=False, encoding='utf-8')
df_articles.to_csv(os.path.join(OUTPUT_ROOT, 'bronze_news_articles.csv'), index=False, encoding='utf-8')
df_stances[df_stances['actor_name'].fillna('').str.strip().ne('')].to_csv(os.path.join(OUTPUT_ROOT, 'gold_news_actor_stances.csv'), index=False, encoding='utf-8')

print(f"[Export Selesai] Folder output: {OUTPUT_ROOT}")
print(f"- Master CSV : {master_csv_path} ({len(df_master):,} baris)")
print(f"- SQLite DB  : {db_path}")
display(df_master.head(5))


## 10. Ringkasan Diagnostik Pasca-Run


In [ ]:
print("=" * 75)
print(f"RINGKASAN RUN | Labeler: {CONFIG.labeler_id}")
print("=" * 75)
print(f"Total Artikel Input     : {len(df_matched):,}")
print(f"Artikel Prediksi Relevan: {int(df_matched['pred_relevancy'].sum()):,}")
print("Breakdown Filter Stage 1:")
print(df_matched['relevancy_filter_tier'].value_counts().to_string())

act_valid = df_master[df_master['actor_name'].fillna('').str.strip().ne('')]
print(f"\nTotal Pernyataan Aktor  : {len(act_valid):,}")
if len(act_valid) > 0:
    grounded = act_valid['start_char'].notna().sum()
    print(f"Verbatim Grounded Rate  : {grounded}/{len(act_valid)} ({grounded/len(act_valid)*100:.1f}%)")
    print("Distribusi Stance (web lowercase):")
    print(act_valid['stance'].value_counts().to_string())
